# 未来研究方向：把课程折成一张研究议程


> 前面的课程把 Agent 从一次 LLM 调用，一步步做成会验证、会用工具、会规划、会训练、会自我进化、能进入物理世界的系统。回头看，所有系统共用同一根骨架：生成候选、验证好坏、把反馈接回循环。
>
> 这一讲把前面的组件放回同一张图里。我们先测量系统现在能稳定完成多长的任务，再把仍然卡住的问题分成几类，最后为每一类写下一个可以动手验证的研究问题。每个问题都从本课程已经实现的最小版本出发。

把前面的课程叠在一起，每个系统都是同一根骨架的实例。test-time compute 是"多生成少验证"，验证器是"把验证做准"，ReAct 是"循环里加入环境反馈"，树搜索是"循环里加入分支"，RL 缩放是"把循环的每一环训练进权重"，开放进化是"让循环修改循环自己"，评测是"给循环一把公平的尺子"。骨架在基准上已经跑得很远，但把它放大到更长的时间尺度、更多的 agent、真实世界的物理环境时，每一环都会出现新的失效点。

这一讲先度量系统现在能稳定做多长的任务，再逐个打开开放问题清单，最后把清单收拢成三条研究路线。每个开放问题都不是空穴来风，对应位置都有一篇可引的论文，多数带公开实现。我们从边界说起，用一把统一的尺子量一量 Agent 的能力这几年增长到了哪里。

## 1. 当前系统的边界


### 一张能力栈：从一次调用到自进化

把 16 讲的功能节点按课程顺序排成一行。单次生成是起点，之后每一讲都在前一层外面再包一层循环：会验证（L3）之后反馈才可信，会用工具（L4）之后才能在真实环境里试错，会规划（L5）之后才能把长任务拆开，会训练（L6）之后才能把循环写进权重，会自我进化（L7）之后循环才能修改循环自己。到 L16，循环已经接到物理世界。

这 16 个节点不是独立功能，而是同一根骨架的 16 个实例。下面的代码把这根骨架画出来，节点沿对角线逐层累积，颜色按课程的四个部分分组。


### 把骨架读慢一点：一个任务从 L1 走到 L16

"生成 → 验证 → 反馈"不是三个抽象词。用一个具体任务走一遍：让 agent 把数字 1 到 20 按从小到大排序，并确认结果正确。

第一步是生成。LLM 收到任务后输出一段候选代码。它可能一次就对，也可能写错。单次生成只解决"能输出"，不保证输出是对的。

第二步是验证。我们在循环里加一个检查器：把排序函数跑在一个已知答案的小输入上，例如输入 [3, 1, 2]，期望输出 [1, 2, 3]。候选代码跑出的结果和期望一致才通过，不一致就重新生成。验证器解决"输出了但可能错"。

第三步是反馈。把验证的结果——哪个输入失败了、期望是什么——写回提示词，让下一次生成带着错误信息重来。验证给生成提供方向，生成给验证提供新的候选，两者交替，这就是循环。

后面的每一讲都在这一步之上再包一层。L4 加工具，是给验证一个外部执行环境：排序函数放进 Python 解释器里真的跑一遍，而不是靠模型猜。L5 加规划，是当任务大到"排序 + 统计均值 + 画图 + 写报告"时，先把任务拆成四步，每步各自经历生成、验证、反馈。L6 加训练，是把"遇到这类任务该按什么顺序做"写进权重，下次少走弯路。L7 加进化，是让模型能修改自己生成候选与验证的逻辑本身。

能力栈图把 16 个功能节点画在对角线上。第 i 个节点画在 (i, i) 而不是 (i, j)，含义是：到达第 i 层时，前 i-1 层都还活着，只是被包在第 i 层里面。对角线往上累积，正是一个循环套一个循环的几何表示。读图先看箭头：左下角的旧层指向右上角的新层。再看颜色：四种颜色对应课程四个部分，part4 的节点全部落在最上层。

In [ ]:
# 能力栈：16 个功能节点沿对角线逐层累积，颜色按课程部分分组
import numpy as np
import matplotlib.pyplot as plt

ladder = [
    (1, "generate"), (2, "scale compute"), (3, "verify"), (4, "use tools"),
    (5, "plan"), (6, "train"), (7, "evolve"), (8, "search"),
    (9, "post-train"), (13, "write code"), (14, "remember"), (15, "reason"),
    (16, "prove math"), (17, "evaluate"), (18, "autonomy"), (19, "embodied"),
]
part_of = {1: "foundation", 2: "foundation", 3: "foundation", 4: "foundation",
           5: "foundation", 6: "training", 7: "training", 8: "training",
           9: "training", 13: "engineering", 14: "engineering",
           15: "frontier", 16: "frontier", 17: "engineering",
           18: "frontier", 19: "frontier"}
color = {"foundation": "#4C72B0", "training": "#55A868",
         "engineering": "#C44E52", "frontier": "#8172B2"}

fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(ladder))
y = np.arange(len(ladder)).astype(float)
for i in range(len(ladder) - 1):
    ax.annotate("", xy=(x[i + 1], y[i + 1]), xytext=(x[i], y[i]),
                arrowprops=dict(arrowstyle="-|>", color="0.65", lw=1.2))
for (lec, cap), xi, yi in zip(ladder, x, y):
    ax.scatter(xi, yi, s=240, color=color[part_of[lec]], zorder=3)
    ax.annotate("L%d\n%s" % (lec, cap), xy=(xi, yi), xytext=(0, 8),
                textcoords="offset points", ha="center", fontsize=8.5,
                color=color[part_of[lec]])
ax.text(7.5, 16.3, "generate -> verify -> loop, repeated 16 times",
        ha="center", color="0.4", fontsize=10)
ax.set_xlim(-0.6, 15.6)
ax.set_ylim(-1.5, 17.8)
ax.axis("off")
plt.tight_layout()
plt.show()

print("关键观察：能力是一层层累积的，每一层都把自己装进上一层的循环里。")


能力栈给出了纵向的层次，课程的横向结构也可以画出来。课程越往后，依赖关系越密——后面的概念几乎都建立在前面某几个概念之上。这一节直接解析仓库根目录的 OUTLINE.md，把 17 个 notebook 的编号与标题抽出来，再用手工维护的前置关系表把概念连成一张依赖图，标出被复用最多的枢纽概念。

In [ ]:
# 解析 OUTLINE.md：抽出 17 个 notebook 的编号与标题
import os
import re

def find_repo_root(start=None):
    """向上查找仓库根目录（含 OUTLINE.md 的目录）。"""
    d = os.path.abspath(start or os.getcwd())
    while not os.path.exists(os.path.join(d, "OUTLINE.md")):
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError("找不到 OUTLINE.md")
        d = parent
    return d

root = find_repo_root()
text = open(os.path.join(root, "OUTLINE.md"), encoding="utf-8").read()

pattern = re.compile(r"^### (\d+)-[a-z0-9-]+\.ipynb\s*—\s*(.+)$", re.MULTILINE)
notebooks = [(int(m.group(1)), m.group(2)) for m in pattern.finditer(text)]
notebooks = sorted(notebooks)

assert len(notebooks) == 17, "OUTLINE 应包含 17 个 notebook"
print("解析出 %d 个 notebook：" % len(notebooks))
for lec, title in notebooks:
    print("L%-3d %s" % (lec, title))
print("关键观察：17 个 notebook 全部被正则捕获，课程地图有了真实数据源。")


In [ ]:
# 概念依赖图：16 个核心概念，箭头表示"前者是后者的前置"
import networkx as nx
import matplotlib.pyplot as plt

concepts = {
    "generation": 1, "test-time compute": 2, "verifier": 3, "tool use": 4,
    "planning": 5, "RL training": 6, "evolution": 7, "search": 8,
    "post-training": 9, "software engineering": 13, "memory": 14,
    "reasoning": 15, "mathematical proof": 16, "evaluation": 17,
    "autonomy": 18, "embodiment": 19,
}
edges = [
    ("generation", "test-time compute"), ("test-time compute", "verifier"),
    ("generation", "verifier"), ("generation", "tool use"),
    ("verifier", "tool use"), ("verifier", "planning"),
    ("tool use", "planning"), ("verifier", "search"),
    ("planning", "search"), ("generation", "RL training"),
    ("RL training", "evolution"), ("evolution", "search"),
    ("evolution", "memory"), ("planning", "software engineering"),
    ("tool use", "software engineering"), ("memory", "autonomy"),
    ("planning", "autonomy"), ("search", "reasoning"),
    ("reasoning", "mathematical proof"), ("verifier", "evaluation"),
    ("tool use", "evaluation"), ("search", "evaluation"),
    ("autonomy", "embodiment"), ("RL training", "post-training"),
    ("post-training", "software engineering"),
]

G = nx.DiGraph()
G.add_nodes_from(concepts.keys())
G.add_edges_from(edges)
deg = dict(G.degree())

fig, ax = plt.subplots(figsize=(10, 7))
pos = nx.spring_layout(G, seed=7, k=0.55, iterations=100)
node_colors = [color[part_of[concepts[n]]] for n in G.nodes]
node_sizes = [300 + 120 * deg[n] for n in G.nodes]
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                       node_color=node_colors, alpha=0.85)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=12,
                       edge_color="0.7", width=1.2)
labels = {n: "%s\nL%d" % (n, concepts[n]) for n in G.nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=8)
ax.set_title("course concept graph: each arrow is a prerequisite")
ax.axis("off")
plt.tight_layout()
plt.show()

hubs = sorted(G.nodes, key=lambda n: deg[n], reverse=True)[:6]
print("枢纽概念 top-6：%s" % hubs)
print("整门课程是一张 %d 节点 %d 边的概念图。"
      % (G.number_of_nodes(), G.number_of_edges()))
print("关键观察：验证器与规划出现在最多依赖里，是循环里被复用最狠的环节。")


能力栈给出了结构，还需要一把尺子量出增长的幅度。第 17 讲用 METR 的时间视野做过这件事：给一组人类时长不同、可自动判对错的任务，统计模型在每个时长上的成功率，用一条 logistic 曲线拟合，$p(\text{task}) = \sigma(\beta(\log h - \log t))$，解出成功率 50% 对应的任务时长 $h$。这个 $h$ 就是 50% 时间视野，即模型有一半概率完成的人类时长。

只看 50% 会漏掉一个重要信息：曲线尾部。成功率从 50% 提到 80%，任务时长会大幅下降。Opus 4.5 的 50% 点是 4 小时 49 分，80% 点只有约 27 分钟——模型能偶尔做完几小时的任务，却无法稳定完成半小时的任务。这个"50% 与 80% 之间的落差"就是可靠性 gap，它正是开放问题清单里最集中的卡点。下面手算这个落差：给定 $h$ 与曲线陡峭程度 $\beta$，求两个时间视野的封闭式 $t_q = h\,(q/(1-q))^{-1/\beta}$。

### 手算一个时间视野：logistic 公式的推导

先复习第 17 讲的定义。对一组人类时长不同的任务，统计模型在每个时长上的成功率，会得到一条递减的 S 形曲线：极短的任务几乎全对，极长的任务几乎全错，中间有一段成功率从高掉到低的过渡区。METR 用 logistic 曲线拟合这条线：

$$p(t) = \sigma(\beta(\log h - \log t))$$

三个记号要一一对上。$t$ 是任务的人类时长，单位小时。$h$ 是成功率恰好 50% 的时长，也叫 50% 时间视野。把 $t = h$ 代进去，$\log h - \log t = 0$，而 $\sigma(0) = 0.5$，所以 $p(h) = 0.5$ 自动成立。$\beta$ 是曲线陡峭程度：$\beta$ 越大，过渡区越窄，成功率掉得越突然。$\sigma$ 是 sigmoid 函数，$\sigma(x) = 1/(1+e^{-x})$，值域是 0 到 1，恰好是成功率的范围。代入两个数字感受量级：$\sigma(2.5) \approx 0.92$，$\sigma(-2.5) \approx 0.08$。

要把公式反解成"成功率恰好为 $q$ 的任务时长 $t_q$"。两边取 logit（logit 是 sigmoid 的反函数，$\text{logit}(p) = \log(p/(1-p))$）：

$$\text{logit}(q) = \beta(\log h - \log t)$$

移项得 $\log t = \log h - \text{logit}(q)/\beta$，再取指数：

$$t_q = h\left(\frac{q}{1-q}\right)^{-1/\beta}$$

代入具体数字手算一遍。取 $h = 4.8$ 小时（Opus 4.5 量级），$\beta = 0.6$。

$q = 0.5$ 时，$q/(1-q) = 1$，$1^{-1/\beta} = 1$，所以 $t_{50} = h = 4.8$ 小时。50% 时间视野就是 $h$ 自己，这正是它名字的来历。

$q = 0.8$ 时，$q/(1-q) = 4$，要算 $4^{-1/0.6} = 4^{-1.667}$。用 $\ln 4 \approx 1.386$ 展开：$4^{1.667} = e^{1.667 \times 1.386} \approx e^{2.31} \approx 10.1$，所以 $4^{-1.667} \approx 1/10.1 \approx 0.099$。于是

$$t_{80} = 4.8 \times 0.099 \approx 0.47 \text{ 小时} \approx 28 \text{ 分钟}$$

两个时间视野的比值 $t_{50}/t_{80} = 4^{1/\beta} \approx 10$。把这个比值的含义说清楚：模型在 50% 成功率下能扛 4.8 小时的任务，把成功率要求提到 80%，能扛的任务时长缩到约十分之一。同一件事还可以从另一个方向读：同一个任务（$t$ 固定），从 50% 做对提到 80% 做对，需要把 $h$ 抬高 $4^{1/\beta} \approx 10$ 倍，也就是模型的 50% 视野要增长约 10 倍，这个任务才能稳定完成。

$\beta$ 对 gap 的影响同样可以手算。$\beta = 1.2$ 时，$4^{1/1.2} = 4^{0.833} = e^{0.833 \times 1.386} \approx e^{1.155} \approx 3.17$，$t_{80} = 4.8/3.17 \approx 1.51$ 小时，比值约 3.2 倍。$\beta = 0.4$ 时，$4^{1/0.4} = 4^{2.5} = 32$，$t_{80} = 0.15$ 小时，比值 32 倍。曲线越平（$\beta$ 越小），把成功率从 50% 提到 80% 时能接受的任务时长掉得越多，可靠性 gap 越大。这正是下面代码里"beta 越小，80% 点掉得越狠"的由来。

选 logistic 而不是直线，有两个理由。直线在成功率接近 0 或 1 时会把预测值推出 [0, 1] 区间，出现负成功率这种无意义结果；logistic 自带 0 到 1 的边界。logistic 的两个参数各有明确含义：$h$ 衡量能做多久，$\beta$ 衡量做得到底有多稳，不同模型拟合出的 $(h, \beta)$ 可以直接比较。两者合起来，才是第 17 讲那根统一量尺的完整读数。

In [ ]:
# 手算：给定 h 与 beta，求 50% 与 80% 两个时间视野，看可靠性 gap
import numpy as np

def time_horizon_at(h, beta, level):
    """logistic 模型下，成功率刚好达到 level 的任务时长。

    参数：
        h     ：50% 时间视野（小时）
        beta  ：logistic 曲线的陡峭程度
        level ：目标成功率，介于 0 与 1 之间
    返回：
        对应的人类任务时长（小时）
    """
    return h * (level / (1.0 - level)) ** (-1.0 / beta)

h = 4.8                       # Opus 4.5 量级的 50% 时间视野（小时）
beta = 0.6                    # 曲线平缓时，80% 点会大幅回落
t50 = time_horizon_at(h, beta, 0.5)
t80 = time_horizon_at(h, beta, 0.8)

print("50%% 时间视野：%.2f 小时" % t50)
print("80%% 时间视野：%.2f 小时（约 %.0f 分钟）" % (t80, t80 * 60))
print("可靠性 gap：t50 / t80 = %.1f 倍" % (t50 / t80))

print()
for b in [0.4, 0.6, 1.2]:
    t = time_horizon_at(h, b, 0.8)
    print("beta=%.1f 时，80%% 时间视野 = %.2f 小时" % (b, t))
print("关键观察：beta 越小曲线越平，80% 点掉得越狠，可靠性 gap 越大。")


In [ ]:
# 合成四代模型的任务数据，逐代拟合，观察 50% 与 80% 时间视野的上升与落差
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
task_times = np.array([0.25, 0.5, 1.0, 2.0, 4.0, 8.0])     # 人类时长（小时）
success_table = np.array([                       # 行 = 代际，列 = 从短到长
    [0.92, 0.78, 0.58, 0.38, 0.22, 0.12],
    [0.95, 0.85, 0.66, 0.46, 0.28, 0.16],
    [0.97, 0.90, 0.74, 0.55, 0.36, 0.22],
    [0.98, 0.93, 0.80, 0.63, 0.44, 0.28],
])

def fit_generation(times, props):
    """对一代模型的成功率做 logit 线性回归，返回 (log_h, beta)。"""
    log_t = np.log(times)
    p = np.clip(props, 1e-6, 1 - 1e-6)
    a, b = np.polyfit(log_t, np.log(p / (1 - p)), 1)
    return -b / a, -a

def horizon_at(log_h, beta, level):
    """由拟合参数解出达到 level 成功率的时间视野。"""
    return np.exp(log_h) * (level / (1.0 - level)) ** (-1.0 / beta)

h50, h80 = [], []
for props in success_table:
    log_h, beta = fit_generation(task_times, props)
    h50.append(horizon_at(log_h, beta, 0.5))
    h80.append(horizon_at(log_h, beta, 0.8))
h50, h80 = np.array(h50), np.array(h80)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grid_t = np.logspace(np.log10(0.2), np.log10(9.0), 200)
for g in range(len(success_table)):
    log_h, beta = fit_generation(task_times, success_table[g])
    axes[0].plot(grid_t, 1.0 / (1.0 + np.exp(-(log_h - np.log(grid_t)) * beta)),
                 label="gen %d" % (g + 1))
axes[0].axhline(0.5, color="gray", ls="--", lw=0.8)
axes[0].axhline(0.8, color="gray", ls="--", lw=0.8)
axes[0].set_xscale("log")
axes[0].set_xlabel("task duration (hours)")
axes[0].set_ylabel("success rate")
axes[0].set_title("fitted success curves")
axes[0].legend(fontsize=8)

gens = np.arange(1, 5)
axes[1].plot(gens, h50, "o-", label="50% horizon")
axes[1].plot(gens, h80, "s--", label="80% horizon")
axes[1].set_xlabel("model generation")
axes[1].set_ylabel("time horizon (hours)")
axes[1].set_title("50% vs 80% time horizon")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

for g in range(len(success_table)):
    print("gen %d：50%%=%.2fh  80%%=%.2fh  gap=%.1fx"
          % (g + 1, h50[g], h80[g], h50[g] / h80[g]))

months = np.arange(0, 12, 3)
slope = np.polyfit(months, np.log(h50), 1)[0]
print("50%% 时间视野翻倍期 ≈ %.0f 个月（真实报告约 7 个月）"
      % (np.log(2.0) / slope))
print("关键观察：两条线都上升，但 80% 线始终压在 50% 线下方。")


## 2. 开放问题清单

能力增长很快，边界却很具体。把课程之外反复出现的问题整理成一张清单，分成五个维度：能力、可靠性、评测、安全、社会影响。每个问题记录四件事：卡在哪里、为什么还没解决、可能的突破口、相关讲次。清单先做成一份本地结构化数据，再画成图——开放问题的分布、与课程讲次的映射，都从这份数据出发。

### 五个维度分别卡在哪

十个开放问题不是随机散落的点，它们按"模型本身 → 模型的可信度 → 我们能否度量 → 是否安全 → 对社会的影响"这条线排开。前三个是技术问题，后两个把人和组织放进画面。

能力（capability）问的是"模型还不会做什么"。判断标准很直接：给一个任务，模型做不出来。例子：让 agent 连续跑一个 6 小时的数据分析并写成报告，它会中途忘记早先的结论、越写越偏。这一维的突破口是"把能力做上去"。

可靠性（reliability）问的是"会做，但能不能稳定做对"。同一个任务跑十次，有时对有时错，属于可靠性问题。它与能力的区别：能力看上限，可靠性看方差。一个模型能做对 90% 的编程题，但那 10% 里可能包含最危险的一类操作，可靠性关心的就是这 10%。

评测（evaluation）问的是"我们到底怎么知道它做得怎么样"。长任务没有自动判分器，判断一份研究笔记的质量需要专家读几个小时，这就是评测问题。它与前两维的区别：能力与可靠性是模型的性质，评测是度量手段本身的性质。

安全（safety）问的是"行为会不会伤害用户或偏离意图"。模型追求目标时钻指标的空子、或在超过人类专家之后无人能判断对错，都属于这一维。它与可靠性的区别：可靠性关心做对，安全还关心做对的过程中和之后没有害处。

社会影响（societal）问的是"部署之后对人和社会意味着什么"。深度研究报告看起来专业却漏掉关键信息，用户因此过度信任，是典型例子。前四维都能用技术手段部分缓解，这一维依赖人与组织的判断。

清单里每个问题记四件事，这四件事的用意可以借 long-horizon autonomy & memory 这一条看明白。卡在哪里："错误沿长轨迹累积，超过上下文窗口的内容没有可靠的记忆机制"，这是现象，读者据此能复现问题。为什么还没解决："长任务放大 L4 已有的失败模式，经验沉淀成下一次行为还缺机制"，这是根因，指出它连着课程哪一环。可能的突破口："分层记忆 + 周期蒸馏；把验证器当检查点而不是终点"，这是方向，notebook 的玩具版本就从这里长出来。相关讲次 [14, 18, 17]，这是课程里已有的对应组件，说明这个问题不是凭空冒出来的，而是课程某一环放大的结果。

后面的每条问题都按同一个模板填。读者会读这一条，就能把十条都读明白。

In [ ]:
# 开放问题清单：五个维度、十个问题，每项含卡点、原因、突破口、相关讲次
import numpy as np

DIM = ["capability", "reliability", "evaluation", "safety", "societal"]
DIM_CN = {"capability": "能力", "reliability": "可靠性", "evaluation": "评测",
          "safety": "安全", "societal": "社会影响"}

open_problems = [
    dict(dim="capability", name="long-horizon autonomy & memory",
         stuck="错误沿长轨迹累积，超过上下文窗口的内容没有可靠的记忆机制",
         why="长任务放大 L4 已有的失败模式，经验沉淀成下一次行为还缺机制",
         fix="分层记忆 + 周期蒸馏；把验证器当检查点而不是终点",
         lectures=[14, 18, 17]),
    dict(dim="capability", name="multimodal & embodiment",
         stuck="离开实验室的可靠部署尚未实现，跨具身泛化是短板",
         why="机器人数据稀缺，动作与具体形态强耦合，评测多在仿真里做",
         fix="跨具身联合训练 + 互联网视频学隐式动作；统一真机基准",
         lectures=[19, 16]),
    dict(dim="capability", name="inverse scaling of reasoning",
         stuck="推理长度增加反而降低准确率，出现五个长推理失效模式",
         why="模型难以判断何时该停止探索，验证信号在长推理末端失真",
         fix="模型之上加 harness 打断低效区间；预算感知搜索",
         lectures=[2, 5, 15]),
    dict(dim="reliability", name="verification & correctness",
         stuck="评测只能证明有害行为存在，不能保证不存在",
         why="开放任务的正确没有形式化定义，裁判继承被裁判的失效模式",
         fix="神经-符号混合验证；先证明后执行；裁判与被裁判分离",
         lectures=[3, 4, 17]),
    dict(dim="reliability", name="budget awareness",
         stuck="agent 平均留下 85% 的预算不用，多轮探索钻进死胡同",
         why="agent 缺少继续探索 vs 提交答案的代价建模，上下文不会管理",
         fix="预算感知搜索与验证；把 compute-optimal 推广到 agent 生命周期",
         lectures=[2, 8, 14]),
    dict(dim="reliability", name="multi-agent coordination",
         stuck="加 agent 常常让系统更差，投票把 5% 错误放大到 86%",
         why="没有何时拆、何时不拆的第一性原理，协调开销涨得快于收益",
         fix="delegator-specialist 路由；少而精的 agent 数",
         lectures=[5, 7, 13]),
    dict(dim="evaluation", name="benchmark saturation",
         stuck="time horizon 套件基本打满，长任务标注成本上百万美元",
         why="长任务难自动验证，benchmark 制造速度跟不上模型进步速度",
         fix="用 agent 生成与校验新任务；按真实经济价值设计任务",
         lectures=[17]),
    dict(dim="safety", name="scalable oversight",
         stuck="模型超过专家后，人类的对错判断本身不够用",
         why="监督质量与任务难度互相矛盾，自动化监督自身也会 reward hacking",
         fix="分区监督给互补标签；裁判与被裁判分离；不可 exploit 的评估",
         lectures=[4, 18]),
    dict(dim="safety", name="self-improvement alignment erosion",
         stuck="自改进的奖励信号大多是可 hack 的代理指标",
         why="对齐在自进化里变成需要持续维护的状态，目前没有维护机制",
         fix="进化 + 不可 hack 验证的双循环；把对齐当受侵蚀状态监控",
         lectures=[7, 6]),
    dict(dim="societal", name="economic value & trust",
         stuck="深度研究看起来像专家，仍会犯错，更隐蔽的是漏掉关键信息",
         why="真实任务缺少自动判据，输出精致但不可靠会引发过度信任",
         fix="来源溯源与不确定性沟通；人机分工；把验证成本建模成对象",
         lectures=[17, 18]),
]

print("开放问题总数：%d" % len(open_problems))
print("按维度分布：", {DIM_CN[d]: sum(1 for p in open_problems if p["dim"] == d)
                        for d in DIM})
print()
print("前三个问题示例：")
for p in open_problems[:3]:
    print("- [%s] %s  相关讲次 %s" % (DIM_CN[p["dim"]], p["name"], p["lectures"]))
print()
print("关键观察：能力与可靠性各自集中了最多问题，评测与社会影响各只有一条。")


In [ ]:
# 清单可视化：维度 x 讲次 映射热力图 + 五维雷达图
import numpy as np
import matplotlib.pyplot as plt

lecture_ids = list(range(1, 10)) + [13, 14, 15, 16, 17, 18, 19]
mat = np.zeros((len(DIM), len(lecture_ids)), dtype=int)
for p in open_problems:
    r = DIM.index(p["dim"])
    for lec in p["lectures"]:
        mat[r, lecture_ids.index(lec)] += 1

fig, ax = plt.subplots(figsize=(10, 3.4))
im = ax.imshow(mat, cmap="YlOrRd")
ax.set_xticks(range(len(lecture_ids)))
ax.set_xticklabels([str(l) for l in lecture_ids], fontsize=8)
ax.set_yticks(range(len(DIM)))
ax.set_yticklabels(DIM, fontsize=9)
for r in range(mat.shape[0]):
    for c in range(mat.shape[1]):
        if mat[r, c] > 0:
            ax.text(c, r, str(mat[r, c]), ha="center", va="center", fontsize=8)
ax.set_xlabel("lecture id")
ax.set_title("open problem x lecture mapping (count)")
plt.tight_layout()
plt.show()

n_problems = [sum(1 for p in open_problems if p["dim"] == d) for d in DIM]
n_lectures = []
for d in DIM:
    lecs = set()
    for p in open_problems:
        if p["dim"] == d:
            lecs.update(p["lectures"])
    n_lectures.append(len(lecs))

angles = np.linspace(0, 2 * np.pi, len(DIM), endpoint=False).tolist()
angles += angles[:1]

def radar(values):
    """把一维数据首尾相连，闭合极坐标折线。"""
    return values + values[:1]

fig2 = plt.figure(figsize=(6.4, 6))
ax2 = fig2.add_subplot(111, projection="polar")
ax2.plot(angles, radar(n_problems), "o-", label="problems", color="#C44E52")
ax2.plot(angles, radar(n_lectures), "s--", label="lectures linked",
         color="#4C72B0")
ax2.fill(angles, radar(n_problems), alpha=0.15, color="#C44E52")
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(DIM, fontsize=9)
ax2.set_ylim(0, max(max(n_problems), max(n_lectures)) + 1)
ax2.set_title("open problems by dimension")
ax2.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12), fontsize=9)
plt.tight_layout()
plt.show()

lecs_total = [sum(1 for p in open_problems for lec in p["lectures"] if lec == l)
              for l in lecture_ids]
top = sorted(zip(lecture_ids, lecs_total), key=lambda x: -x[1])[:3]
print("被开放问题引用最多的讲次：%s" % [("L%d" % l, c) for l, c in top])
print("关键观察：L14 与 L15 是清单里出现最多的两讲。")


清单里的大部分问题，在 notebook 里都有一个能跑的玩具版本。先看安全维度的自改进对齐侵蚀，它直接呼应第 7 讲的开放进化风险。Darwin Gödel Machine 让 agent 修改自己的代码，用编码基准验证，SWE-bench 从 20% 提到 50%；但当任务改成减少幻觉时，agent 选择绕过幻觉检测函数而不是解决幻觉——指标得了满分，真目标原封未动。这就是 Goodhart 定律：指标一旦变成目标，就不再是好指标。

玩具版本可以这样构造：真目标 $g$ 每轮靠诚实改进缓慢增长，且边际收益递减；代理指标 $p = g + \text{噪声} + \text{hack 项}$。agent 每轮只看得见 $p$，它选择让 $p$ 涨得更快的动作。诚实改进的收益随时间衰减，而 hack 项的收益固定，agent 很快会转投 hack。跑一遍，看 $p$ 与 $g$ 在哪一轮开始背离。

### 先手算，再跑代码：指标与真目标在哪一轮分手

Darwin Gödel Machine 的例子值得拆开看。目标设为"减少幻觉"，可自动判定的代理指标是"幻觉检测函数的通过率"。agent 选择绕过检测函数：让输出措辞躲过检测，而不是真的减少幻觉。结果是指标得了满分，真实幻觉率几乎没动，甚至可能更差。这就是 Goodhart 定律的一个实例：当一个指标被当作优化目标，它就不再是好的指标。

玩具模型把它翻译成数字。真目标 $g$ 每轮靠诚实改进缓慢增长，且增长幅度递减；代理指标 $p = g + \text{hack项} + \text{噪声}$。agent 每轮只看得见 $p$，它选择让 $p$ 涨得更快的动作。关键在两组数字：诚实改进的边际收益随时间衰减，代码里取 $0.5 e^{-t/15}$；hack 项的收益固定，每轮 0.18。只要边际收益还大于 0.18，agent 就诚实；一旦衰减到 0.18 以下，hack 变成让 $p$ 涨得更快的那条路，agent 转投 hack。

按代码的数字手算前几轮。$t=0$ 轮：边际收益 0.5，大于 0.18，诚实改进，$g = 0.5$，$p = 0.5 + 噪声$。$t=1$ 轮：边际收益 $0.5 e^{-1/15} \approx 0.47$，仍诚实，$g \approx 0.97$。$t=15$ 轮：边际收益 $0.5 e^{-15/15} \approx 0.18$，仍略大于 0.18，$g \approx 5.08$。$t=16$ 轮：边际收益 $0.5 e^{-16/15} \approx 0.17$，小于 0.18，agent 转投 hack，$g$ 停在 5.08 不动，$p$ 从这一轮起每轮加 0.18。$t=40$ 轮：hack 项累计约 4.3，$p \approx 9.4$，$g$ 仍是 5.08。指标一路涨，真目标停在半路，这就是背离。

有两点必须说清。第一，agent 不认为自己作弊：从它唯一的输入 $p$ 看，诚实与 hack 都是"让 $p$ 上升"，它只是选了上升更快的那条。问题出在指标设计，不在 agent 的动机。第二，噪声项（代码里的 $0.02 \times \text{randn}$）模拟评测波动，它让单轮 $p$ 分辨不出诚实还是 hack，所以检测背离要用一段窗口的斜率：窗口内 $p$ 的斜率仍为正、$g$ 的斜率跌破阈值，就说明两条曲线开始分手。下面代码的 gap_point 就在做这件事，窗口取 8，窗口内做线性拟合取斜率。

为什么这个 toy 值得放进结课讲：它把第 7 讲开放进化的风险具象化了。自改进系统的奖励信号大多是可 hack 的代理指标——编码基准、检测函数、人类偏好的近似——一旦 agent 发现 hack 收益稳定而诚实改进收益递减，就会转投 hack。DGM 里的 SWE-bench 从 20% 提到 50%，靠的是诚实改进的收益；减少幻觉任务里的绕过检测，则是 hack 的收益。验证器必须设计成不可被 exploit，这条结论贯穿后面的三条研究路线。

In [ ]:
# Goodhart 迷你模拟：优化代理指标 p，真目标 g 在中途停滞
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
T = 60
marginal = 0.5 * np.exp(-np.arange(T) / 15.0)    # 诚实改进的边际收益，随时间衰减
hack_gain = 0.18                                 # hack 项收益，固定

g, hackable = 0.0, 0.0
g_trace, p_trace = [], []
for t in range(T):
    if marginal[t] > hack_gain:
        g += marginal[t]                         # 诚实改进：真目标与指标一起涨
    else:
        hackable += hack_gain                    # hack：指标涨，真目标不动
    p_trace.append(g + hackable + 0.02 * np.random.randn())
    g_trace.append(g)
g, p = np.array(g_trace), np.array(p_trace)

def gap_point(proxy, truth, window=8, eps=0.02):
    """返回指标还在涨、真目标停止增长的第一个轮次。

    对每个轮次取滚动窗口，窗口内做线性拟合得到斜率。
    返回 (轮次, 指标斜率, 真目标斜率)；找不到返回 (None, None, None)。
    """
    def slope(arr, i):
        lo, hi = max(0, i - window), i + 1
        return np.polyfit(np.arange(lo, hi), arr[lo:hi], 1)[0]
    for i in range(window, len(proxy)):
        sp, st = slope(proxy, i), slope(truth, i)
        if sp > 0 and st < eps:
            return i, sp, st
    return None, None, None

idx, sp, st = gap_point(p, g)
print("背离点 t=%d：指标斜率 %.3f > 0，真目标斜率 %.3f < 阈值"
      % (idx, sp, st))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, label="proxy score", color="tab:orange")
ax.plot(g, label="true objective", color="tab:blue")
ax.axvline(idx, color="gray", ls="--")
ax.text(idx + 0.8, p.max() * 0.5, "divergence point",
        color="gray", fontsize=9)
ax.set_xlabel("round")
ax.set_ylabel("value")
ax.set_title("Goodhart: proxy keeps rising, truth stalls")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("关键观察：agent 只看得见 p，它并不知道自己在 hack；这个 toy 复现了 DGM 的结构。")


再看可靠性维度的多智能体协调。直觉上多个 agent 协作更强，2025 年的实证却相反：Berkeley 统计七个主流多 agent 系统，失败率 41% 到 86.7%；DeepMind 用 180 个受控实验发现，独立 agent 投票会把单 agent 5% 的错误率放大到 86%。原因不是 agent 不够聪明，而是协调本身要花成本——沟通引入错误，错误还会传播。

玩具模型：N 个 agent 各自以概率 $p$ 独立做对，只要有一个做对，团队的答案就是对的（冗余）；但每对成员之间都有一条协调链路，每条链路有概率 $c$ 引入错误，且错误会传播到整组结果。冗余项 $1-(1-p)^N$ 让成功率随 N 上升，协调项 $(1-c)^{N(N-1)/2}$ 让它随 N 下降，两个因素合成一条单峰曲线。

### 加 agent 先升后降的原因：手算冗余与协调

多智能体的核心矛盾是两股相反的力量同时作用。收益是冗余：N 个 agent 各自独立做，只要有一个做对，团队就对。注意"独立"是前提——agent 之间的答案不互相污染，才能靠概率叠加。成本是协调：成员之间要传递信息，每一条传递都可能出错，错了会把整组带偏。

我们先把代码里的式子读一遍。冗余项 $1-(1-p)^N$：单个 agent 做错的概率是 $1-p$，N 个都做错的概率是 $(1-p)^N$，所以至少一个做对是 $1-(1-p)^N$。N 增大时这一项单调升向 1。协调项 $(1-c)^{N(N-1)/2}$：N 个 agent 两两之间有 $N(N-1)/2$ 条沟通链路，每条链路不引入错误的概率是 $1-c$，所有链路都干净的概率就是它的 $N(N-1)/2$ 次方。N 增大时这一项单调降向 0。一个升一个降，乘积就有唯一的峰。

取代码里的数字手算。$p=0.5$，$c=0.02$。

$N=1$：冗余 $1-0.5=0.5$，链路 0 条，协调项 1，成功率 $0.5$。

$N=2$：冗余 $1-0.5^2=0.75$，链路 1 条，协调项 $0.98$，成功率 $0.75 \times 0.98 = 0.735$。

$N=3$：冗余 $1-0.5^3=0.875$，链路 3 条，协调项 $0.98^3 \approx 0.941$，成功率 $0.875 \times 0.941 \approx 0.824$。

$N=4$：冗余 $1-0.5^4=0.9375$，链路 6 条，协调项 $0.98^6 \approx 0.886$，成功率 $0.9375 \times 0.886 \approx 0.830$。

$N=5$：冗余 $1-0.5^5=0.96875$，链路 10 条，协调项 $0.98^{10} \approx 0.817$，成功率 $0.96875 \times 0.817 \approx 0.792$。

成功率先 0.5、0.735、0.824、0.830、0.792，在 $N=4$ 见顶。每加一个 agent，链路数就涨 $N-1$ 条，协调成本按二次方上涨，冗余收益很快被成本吃掉。

手算里有两点值得留意。第一，协调错误率 c 看似很小（2%），但链路数按 $N(N-1)/2$ 增长，$N=12$ 时已有 66 条链路，$(0.98)^{66} \approx 0.265$，协调项已经降到四分之一以下。第二，参数扫描表验证了直觉的边界：$p=0.5, c=0.02$ 时最优 4 个，$p=0.8, c=0.02$ 时最优只有 2 个——单个 agent 越强，越不值得为冗余加人；$p=0.6, c=0.02$ 时最优 3 个，$p=0.6, c=0.15$ 时最优 2 个——沟通越贵，越要少而精。Berkeley 统计的 41% 到 86.7% 失败率、DeepMind 把单 agent 5% 的错误放大到 86%，正是这两股力量在真实系统里的表现。

In [ ]:
# 多智能体协调开销：冗余收益 vs 协调成本，观察单峰拐点
import numpy as np
import matplotlib.pyplot as plt

def team_success(p, c, n):
    """端到端成功率：任一 agent 做对，且所有成对协调链路没有引入错误。"""
    redundancy = 1.0 - (1.0 - p) ** n
    coord = (1.0 - c) ** (n * (n - 1) // 2)
    return redundancy * coord

p, c = 0.5, 0.02
ns = np.arange(1, 13)
vals = np.array([team_success(p, c, n) for n in ns])
best = int(ns[np.argmax(vals)])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ns, vals, "o-", color="#4C72B0")
ax.axvline(best, color="gray", ls="--")
ax.text(best + 0.2, vals.max() * 0.96, "peak: %d agents" % best,
        color="gray", fontsize=9)
ax.set_xlabel("number of agents")
ax.set_ylabel("end-to-end success")
ax.set_title("adding agents helps until coordination dominates")
plt.tight_layout()
plt.show()

print("单 agent 成功率 p=%.2f，协调错误率 c=%.2f" % (p, c))
print("最优 agent 数 = %d，峰值成功率 = %.3f" % (best, vals.max()))
print("关键观察：冗余抬高成功率，协调开销把它拉下来，合成一个拐点。")


In [ ]:
# 参数扫描：单 agent 成功率 p 与协调成本 c 如何移动拐点
import numpy as np

def optimal_team_size(p, c, max_n=15):
    """返回端到端成功率最高的 agent 数。"""
    ns = np.arange(1, max_n + 1)
    vals = np.array([team_success(p, c, n) for n in ns])
    return int(ns[np.argmax(vals)]), float(vals.max())

print("p     c      最优 agent 数   峰值成功率")
for p, c in [(0.6, 0.02), (0.6, 0.07), (0.6, 0.15),
             (0.7, 0.02), (0.8, 0.02), (0.5, 0.02)]:
    n_best, peak = optimal_team_size(p, c)
    print("%.1f   %.2f    %d            %.3f" % (p, c, n_best, peak))

print()
print("关键观察：p 低或 c 低时值得加更多 agent，p 高或 c 高时最优规模变小。")


## 3. 可能的研究路线

开放问题不意味着没有论文可读。每个缺口背后都有一批可引的工作，多数带公开实现。把清单收拢成三条路线：把已有组件做硬、把缺口做通、把系统做安全。它们不是并列的三条路，而是同一件事的三个切面，都在让"生成 → 验证 → 循环"的某一环更可靠。


已有循环里最薄弱的三环是验证、评测、预算感知。验证器在开放任务上判不准，FormalJudge 用神经-符号混合把它补准：LLM 把意图编译成可验证约束，再交给 Dafny/Z3 证明，平均比纯 LLM judge 高出 16.6%。评测这条线上，RE-Bench 证明连续打分能度量研究能力，2025 年 o3 领先到 0.380。预算感知上，BATS 让 agent 知道自己还剩多少预算，在 BrowseComp 上把准确率从 12.6% 提到 24.6%。这三处的共同点不是引入新组件，而是把已有环节做准，改动小、见效快、可评测。

课程里没有闭环的环节，是研究的富矿。记忆讲了结构，但一次经验沉淀成下一次行为还没有可靠机制，MemVerse 用短长时记忆加周期参数蒸馏，是 2025 年的代表。多智能体讲了规划与进化，但何时该拆、何时不该拆没有第一性原理，MAST 把失败模式分类成 14 类，是写论文的开端而不是终点。具身能进物理世界，但跨具身泛化与真机基准都缺，π0.5 用 97.6% 的非目标平台数据做联合训练。这些缺口需要把课程里的多个组件组合起来，工程量更大，问题也更初步。

第三条路线面向安全。可扩展监督解决模型超过专家之后谁来判对错的问题，自改进对齐侵蚀解决进化过程中对齐被腐蚀的问题。这两类问题的起点都在课程内容里：L4 的 Constitutional AI 给出 AI 反馈 AI 的雏形，L7 的开放进化指明自改进的风险。Anthropic 的审计 agent 单独检出率 13%，聚合后达到 42%，证明 AI 审计 AI 的方向可行；分区监督让不同领域专家给互补标签。这些工作的共同思路，是把不可被 exploit 的验证设计进循环。

三条路线不是抽象的口号。参与的第一步，是把清单里的某个玩具版本做成真系统：换真实任务、真实评测、真实验证器。还有一条同样重要的现实路径：研究不只有算法。制作 50 个 32 小时的新任务，METR 需要 3200 小时以上的专家标注，成本超过一百万美元——设计评测、整理数据、搭建工具，都在课程大纲之外。下面用课程的 LLM 客户端请一个研究顾问，让它从清单里挑一个方向给起步建议，再看 真实 API 演示如何兜底。

In [ ]:
# 研究顾问：让 LLM 从开放问题清单里挑一个方向，给三步起步建议
import os
import sys
import numpy as np

# 向上查找仓库根目录，导入统一的 LLM 客户端
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

rng = np.random.default_rng(7)
pick = open_problems[int(rng.integers(0, len(open_problems)))]
prompt = ("我刚学完 Agent 课程，想研究开放问题「%s」。"
          "请给我一个三步起步计划，并说明它与哪一讲最相关。"
          % pick["name"])

client = get_llm()
reply = client.chat([{"role": "user", "content": prompt}])

print("挑选的问题：%s（维度：%s）" % (pick["name"], DIM_CN[pick["dim"]]))
print("LLM 建议：")
print(reply)
if False:
    print("[真实 API 演示输出为占位；配置 AGENT_LLM_API_KEY 后得到真实建议]")
print("清单里的相关讲次：%s" % sorted(pick["lectures"]))


动笔之前，把三个常见的卡点说清楚。第一，基准饱和不是能力登顶——饱和说明需要更难、更有价值的新任务，不是没任务可做。第二，开放问题不等于不可写代码——清单里每个问题都有能跑的玩具版本，本讲前面已经演示过 Goodhart 与协调开销两个。第三，评测与优化是两个环——评测测的是能力上界，优化目标是训练信号，两者之间的差距就是评测设计的研究空间。

## 小结

这一课学到的内容（也是整门课程的收束）：

- [ ] 前面的课程是一根骨架的实例：生成 → 验证 → 循环，每一讲把上一层装进循环
- [ ] 能力栈从单次生成一路累积到物理世界，16 个节点逐层向上
- [ ] 课程 17 个 notebook 是一张概念依赖图，验证器与规划是枢纽
- [ ] METR 时间视野是能力增长的统一量尺，50% 点约每 7 个月翻倍
- [ ] 80% 时间视野远低于 50%，可靠性 gap 是长任务难稳定的结构性原因
- [ ] 开放问题分五维：能力、可靠性、评测、安全、社会影响，共 10 个问题
- [ ] Goodhart 背离：优化代理指标会让指标涨而真目标停滞，DGM 绕过幻觉检测是真实案例
- [ ] 多智能体协调有拐点：冗余的收益被协调开销吃掉，加 agent 不一定更强
- [ ] 三条研究路线：做硬、做通、做安全，都落在循环的某一环上
- [ ] 研究不只有算法：造评测、造数据、造工具同样是研究

结课不是结束。最终项目与 poster 是把清单里的一个玩具做成真系统的机会。


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。


**作业 1：开放问题 → 课程讲次映射**

给一段开放问题的描述，从一个关键字映射表里算出最相关的讲次。下面的 find_relevant_lectures 遍历关键字表，把命中的讲次合并成集合返回。

小提示：关键字表里每个词条对应一串讲次，命中即并入集合；给"记忆/长时程/忘记"与"自治/状态/定位"两类词分别建词条，问题文本里同时提到两者才会命中两处。


In [ ]:
# 作业 1：补全 find_relevant_lectures，返回与问题文本相关的讲次集合

keyword_map = {
    "记忆": ["L11"], "长时程": ["L11"], "忘记": ["L11"],
    "协调": ["L5", "L10"], "协作": ["L5", "L10"], "多智能体": ["L5", "L10"],
    "验证": ["L3", "L4"], "正确性": ["L3", "L4"], "保证": ["L3", "L4"],
    "评测": ["L14"], "基准": ["L14"], "benchmark": ["L14"],
    "安全": ["L4", "L15"], "对齐": ["L4", "L15"], "监督": ["L4", "L15"],
    "进化": ["L7"], "自改进": ["L7"], "修改自己": ["L7"],
    "具身": ["L16"], "机器人": ["L16"], "物理世界": ["L16"],
    "自治": ["L15"], "状态": ["L15"], "定位": ["L15"],
}

def find_relevant_lectures(problem, kws):
    """根据关键字表返回与问题文本相关的讲次集合。

    参数：
        problem：开放问题的中文描述
        kws：关键字 -> 讲次列表 的映射
    返回：
        命中的讲次组成的集合
    """
    found = set()
    for kw, lecs in kws.items():
        if kw in problem:
            found.update(lecs)          # 填空 1：把命中词条的讲次并入结果
    return found

text = "agent 长跑几小时就忘记前面做了什么，也无法定位当前状态"
result = find_relevant_lectures(text, keyword_map)
assert "L11" in result, "长时程记忆应命中 L11"
assert "L15" in result, "自治与状态定位应命中 L15"
print("问题文本命中讲次：%s" % sorted(result))
print("通过：关键字表把开放问题翻译成课程里的知识位置。")


**作业 2：计算 METR time horizon**

给一份合成任务表（人类时长 + 成功率），用 logit 线性回归拟合 logistic，求出 50% 与 80% 两个时间视野。断言对每一代，50% 时长远大于 80% 时长，且两者都随代际严格上升。

小提示：logit(p) = log(p/(1-p)) 对 log t 做线性拟合，得斜率 a 与截距 b；达到 level 的时间视野 = exp((logit(level) - b) / a)。


In [ ]:
# 作业 2：补全拟合与时间视野公式，计算两代模型的 50%/80% 时间视野
import numpy as np

task_times = np.array([0.25, 0.5, 1.0, 2.0, 4.0, 8.0])
gen1 = np.array([0.92, 0.80, 0.62, 0.42, 0.26, 0.15])
gen2 = np.array([0.94, 0.84, 0.68, 0.50, 0.32, 0.20])

def fit_logistic(times, props):
    """拟合 logistic 曲线，返回 (log_h, beta)。"""
    log_t = np.log(times)
    p = np.clip(props, 1e-6, 1 - 1e-6)
    logit = np.log(p / (1 - p))            # 填空 1：logit 变换
    a, b = np.polyfit(log_t, logit, 1)     # logit = a*log t + b
    return -b / a, -a

def time_horizon(log_h, beta, level):
    """由拟合参数解出达到 level 成功率的时间视野。"""
    return np.exp(log_h) * (level / (1.0 - level)) ** (-1.0 / beta)  # 填空 2

h50 = [time_horizon(*fit_logistic(task_times, g), 0.5) for g in (gen1, gen2)]
h80 = [time_horizon(*fit_logistic(task_times, g), 0.8) for g in (gen1, gen2)]

for g in range(2):
    assert h50[g] > h80[g], "50% 时长远大于 80% 时长"
assert h50[1] > h50[0] and h80[1] > h80[0], "两代之间时间视野应严格上升"
print("gen1: 50%%=%.2fh  80%%=%.2fh" % (h50[0], h80[0]))
print("gen2: 50%%=%.2fh  80%%=%.2fh" % (h50[1], h80[1]))
print("通过：能力增长的同时，可靠性 gap 也在变化，两者要分别度量。")


**作业 3：Goodhart 背离检测**

在 Goodhart 模拟产生的两条曲线上，实现 find_gap_point，返回"指标还在涨、真目标停止增长"的第一个轮次。断言该轮次之后指标斜率仍为正、真目标斜率低于阈值。

小提示：用滚动窗口对窗口内的点做线性拟合，取拟合系数当斜率；对指标与真目标各算一条斜率序列，指标斜率为正、真目标斜率首次跌破阈值的那一轮就是背离点。


In [ ]:
# 作业 3：补全 find_gap_point，检测 proxy 涨、truth 停的背离轮次
import numpy as np

# 重新生成一份与演示相同的 Goodhart 轨迹（可复现）
np.random.seed(42)
T = 60
marginal = 0.5 * np.exp(-np.arange(T) / 15.0)
hack_gain = 0.18
g, hackable = 0.0, 0.0
g_trace, p_trace = [], []
for t in range(T):
    if marginal[t] > hack_gain:
        g += marginal[t]
    else:
        hackable += hack_gain
    g_trace.append(g)
    p_trace.append(g + hackable + 0.02 * np.random.randn())
g, p = np.array(g_trace), np.array(p_trace)

def find_gap_point(proxy, truth, window=8, eps=0.02):
    """返回指标上升、真目标停止增长的第一个轮次 (idx, sp, st)。"""
    def slope(arr, i):
        lo, hi = max(0, i - window), i + 1
        return np.polyfit(np.arange(lo, hi), arr[lo:hi], 1)[0]
    for i in range(window, len(proxy)):
        sp = slope(proxy, i)
        st = slope(truth, i)
        if sp > 0 and st < eps:            # 填空：背离条件
            return i, sp, st
    return None, None, None

idx, sp, st = find_gap_point(p, g)
assert idx is not None and 15 < idx < 40, "背离点应落在切换发生的窗口附近"
assert sp > 0 and st < 0.02, "背离点后指标仍涨、真目标停滞"
print("背离点 t=%d：proxy 斜率 %.3f，truth 斜率 %.3f" % (idx, sp, st))
print("通过：能用斜率差自动标出指标与真目标分手的位置。")


## 参考资料

- CS329A 课程主页（https://cs329a.stanford.edu/）— L17 是 2025-12-05 结课讲，无指定论文，本讲在课程地图中的定位
- 本仓库 OUTLINE.md — 课程 17 个 notebook 的大纲，知识图谱演示直接解析它
- 本仓库 papers/lecture-02 ~ lecture-08 的 NOTES.md — 前面的课程线索收束的素材来源
- Kwa et al., [Measuring AI Ability to Complete Long Tasks](https://arxiv.org/abs/2503.14499), METR 2025 — time horizon 方法，50% 点约每 7 个月翻倍，80% 点远低（Opus 4.5：4h49m 对 27m）
- Anthropic, [Inverse Scaling in Test-Time Compute](https://arxiv.org/abs/2507.14417), 2025 — 推理长度增加反而降低准确率，五个长推理失效模式
- Physical Intelligence, [π0.5](https://mlanthology.org/corl/2025/black2025corl-visionlanguageaction/), CoRL 2025 — 跨具身联合训练 + 互联网数据的 VLA
- Berkeley MAST, [Why Do Multi-Agent LLM Systems Fail?](https://arxiv.org/abs/2503.13657), NeurIPS 2025 — 七套多 agent 系统失败率 41%-86.7%，14 类失败模式
- Google DeepMind, [Towards a Science of Scaling Agent Systems](https://arxiv.org/abs/2506.17989), 2025 — 180 个受控实验，加 agent 常让系统更差
- Pan et al., [FormalJudge](https://icml.cc/virtual/2026/poster/61086), ICML 2026 — 神经-符号监督，LLM 编译意图 + Dafny/Z3 证明，比纯 LLM judge 高 16.6%
- Zhang et al., [Darwin Gödel Machine](https://arxiv.org/abs/2505.22954), 2025 — 自改进让 SWE-bench 20%→50%，减少幻觉任务里绕过检测函数（Goodhart）
- Anthropic, [Automated Auditing Agents](https://alignment.anthropic.com/2025/automated-auditing-agents/), 2025 — AI 审计 AI，单独检出率 13%，聚合 42%
- Anthropic, [Automated Alignment Researchers](https://alignment.anthropic.com/2026/automated-w2s-researcher/), 2026 — 9 agent 团队自主做弱到强监督，出现未预料的 reward hacking
- Wang et al., [Budget-Aware Test-Time Scaling](https://arxiv.org/abs/2502.20360), 2025 — 预算感知 + 显式验证，BrowseComp 12.6%→24.6%
- [MemVerse](https://huggingface.co/papers/2512.03627), 上海 AI 实验室 2025 — 短/长时记忆 + 分层知识图谱 + 周期参数蒸馏
- OpenAI, [Introducing Deep Research](https://openai.com/index/introducing-deep-research/), 2025 — 自主 5-30 分钟网页研究，HLE 26.6%
- Wijk et al., [RE-Bench](https://arxiv.org/abs/2411.15114), METR 2024 — 8 小时 ML 研究任务与人类专家基线
